# Tail-based Task 5 behavior submissions

Keep the scored Empirical Tail v20 as the fixed reference. H1 uses 80% Tail percentile rank and 20% saved peer-change rank. H2 uses 80% of H1 and 20% saved bot-regularity rank. The 80/20 weights are fixed in advance; no search, Kaggle score, or anomaly label is used. Both candidates retain v20 IDs, Tasks 1–4, `1 = anomaly`, and exactly 6,531 alerts. The older v18/v19 files remain immutable historical drafts.

In [1]:
from pathlib import Path
import sys, json, hashlib, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "pipeline" / "save_model.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
SRC = ROOT / "src"
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from src.pipeline.random_seed import set_seed
SEED = set_seed()
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
BASE = SRC / "submission/kaggle_t12345_t5_tail_engineered_v20.csv"
base = pd.read_csv(BASE, dtype=str)
base_card = json.loads(BASE.with_suffix(".json").read_text(encoding="utf-8"))
assert base_card["output_sha256"] == sha(BASE)
sample = pd.read_csv(SRC / "submission/sample_submission.csv", usecols=["id"], dtype=str)
assert base.columns.tolist() == ["id", "task1", "task2", "task3", "task4", "task5"]
assert len(base) == len(sample) == 25889 and base.id.tolist() == sample.id.tolist()
assert base.id.is_unique and base.task5.astype(int).sum() == 6531
print("Frozen Tail v20", sha(BASE), "alerts", base.task5.astype(int).sum())

Frozen Tail v20 2dd79b546af197688d3ff35211cc744c2e86a28360ea3a01343f743364c1505a alerts 6531


In [2]:
def read_test(name):
    path = SRC / "task5/data/eda" / f"{name}.json"
    card = json.loads(path.read_text(encoding="utf-8"))
    assert set(card["splits"]) == {"test"}
    frame = pd.read_parquet(SRC / card["splits"]["test"]["path"])
    assert frame.player_id.is_unique and "is_anomaly" not in frame
    return frame, path
old, old_card = read_test("eda_1_v2")
new, new_card = read_test("eda_behavior_views_v3")
assert old.player_id.equals(new.player_id) and len(new) == len(base)
def saved_model(name, data_name):
    path = SRC / "task5/save" / f"{name}.json"
    card = json.loads(path.read_text(encoding="utf-8"))
    assert card["name"] == name and card["data_name"] == data_name
    return joblib.load(SRC / card["model_path"]), path
tail, tail_card = saved_model("t5_r2_tail_engineered_v2", "eda_1_v2")
peer, peer_card = saved_model("t5_peer_change_v3", "eda_behavior_views_v3")
bot, bot_card = saved_model("t5_bot_regular_v3", "eda_behavior_views_v3")
old_X, new_X = old.drop(columns="player_id"), new.drop(columns="player_id")
print("Saved model and EDA lineage verified")

Saved model and EDA lineage verified


In [3]:
def rank(values):
    values = np.asarray(values, dtype=float)
    assert values.shape == (len(base),) and np.isfinite(values).all()
    return rankdata(values, method="average") / len(values)
def select(values, k=6531):
    chosen = np.lexsort((np.arange(len(values)), -np.asarray(values)))[:k]
    pred = np.zeros(len(values), dtype=np.int8)
    pred[chosen] = 1
    return pred
tail_rank = rank(tail.anomaly_score(old_X))
peer_rank = rank(peer.anomaly_score(new_X))
bot_rank = rank(bot.anomaly_score(new_X))
assert np.array_equal(select(tail_rank), base.task5.astype(np.int8).to_numpy())
h1_rank = 0.8 * tail_rank + 0.2 * peer_rank
h2_rank = 0.8 * h1_rank + 0.2 * bot_rank
h1, h2 = select(h1_rank), select(h2_rank)
assert int(h1.sum()) == int(h2.sum()) == 6531
for label, pred, ref in [("H1 vs Tail", h1, base.task5.astype(np.int8).to_numpy()), ("H2 vs H1", h2, h1)]:
    added = int(((pred == 1) & (ref == 0)).sum())
    removed = int(((pred == 0) & (ref == 1)).sum())
    assert added == removed
    print(label, "swap in/out", added)
print("Score Spearman: Tail/peer", round(spearmanr(tail_rank, peer_rank).statistic, 4), "H1/bot", round(spearmanr(h1_rank, bot_rank).statistic, 4))

H1 vs Tail swap in/out 783
H2 vs H1 swap in/out 974
Score Spearman: Tail/peer 0.2766 H1/bot -0.0179


In [4]:
specs = [
    ("kaggle_t12345_t5_tail_peer_v22", h1, base.task5.astype(np.int8).to_numpy(),
     "H1: peer-relative change adds information to Empirical Tail",
     "0.8*tail_rank + 0.2*peer_rank", BASE.name, [tail_card, peer_card]),
    ("kaggle_t12345_t5_tail_peer_bot_v23", h2, h1,
     "H2: mechanical regularity adds information beyond Tail plus peer change",
     "0.8*h1_rank + 0.2*bot_rank", "kaggle_t12345_t5_tail_peer_v22.csv", [tail_card, peer_card, bot_card]),
]
for name, pred, reference, hypothesis, formula, comparison, model_cards in specs:
    csv = SRC / "submission" / f"{name}.csv"
    meta = csv.with_suffix(".json")
    assert csv.exists() == meta.exists(), f"Incomplete destination: {name}"
    output = base.copy()
    output["task5"] = pred.astype(str)
    assert output[["id", "task1", "task2", "task3", "task4"]].equals(base[["id", "task1", "task2", "task3", "task4"]])
    assert output.notna().all().all() and int(output.task5.astype(int).sum()) == 6531
    added = int(((pred == 1) & (reference == 0)).sum())
    removed = int(((pred == 0) & (reference == 1)).sum())
    assert added == removed
    detail = {
        "name": name, "date_prepared": "2026-11-15", "status": "prepared locally; not uploaded",
        "base_submission": BASE.name, "base_sha256": sha(BASE),
        "comparison_submission": comparison, "changed_task": "task5", "hypothesis": hypothesis,
        "score_formula": formula, "alert_budget": 6531, "alerts": int(pred.sum()),
        "swapped_in_vs_comparison": added, "swapped_out_vs_comparison": removed,
        "task5_orientation": "1 = anomaly", "rows": len(output),
        "model_cards": [p.relative_to(SRC).as_posix() for p in model_cards],
        "dataset_cards": [p.relative_to(SRC).as_posix() for p in [old_card, new_card]],
        "seed": SEED,
    }
    if csv.exists():
        saved = pd.read_csv(csv, dtype=str)
        existing = json.loads(meta.read_text(encoding="utf-8"))
        assert saved.equals(output) and existing["output_sha256"] == sha(csv)
        assert all(existing[k] == v for k, v in detail.items())
    else:
        output.to_csv(csv, index=False)
        detail["output_sha256"] = sha(csv)
        meta.write_text(json.dumps(detail, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("Verified", csv.name, "swap", added, "SHA-256", sha(csv))

Verified kaggle_t12345_t5_tail_peer_v22.csv swap 783 SHA-256 41a8a2eef3e488400b44653642f9cc579f67afa15f53e5d206a9dc618d368a89
Verified kaggle_t12345_t5_tail_peer_bot_v23.csv swap 974 SHA-256 538a09a07779b2ae46bf75606efb7e8f6135cb98318f2b5e3f42d1928de99682
